In [ ]:
# Notebook setup
import sys
from pathlib import Path
base = Path.cwd()
source = next((p for p in (base, *base.parents) if (p / "code/pyproject.toml").is_file()), base / "Release-Client.zip")
if not source.exists() and "google.colab" in sys.modules:
    from google.colab import files
    print("Required file: Release-Client.zip (project code). Choose this ZIP; data files are requested next.", flush=True)
    files.upload(target_dir=str(base))
if not source.exists():
    raise FileNotFoundError("Supply Release-Client.zip, or open the extracted code/notebooks folder.")
sys.path.insert(0, str(source / "cost-aware-market-forecasting" if source.is_file() else source))
sys.modules.pop("run_zip", None)
from run_zip import prepare_reader
CODE = CODE_ROOT = prepare_reader('17_RQ3_F_indices_Q2_sentiment_sensitivity.ipynb', source, ())

# 17 - RQ3_F: Index Q2 sentiment-feed sensitivity

## Methodology

The same four frozen index policies are replayed over April-June 2026 with and without fresh Q2 sentiment. Models, thresholds, price/VIX inputs and execution remain unchanged, with 2 bps round-trip costs for USA500 and 3 bps for USATECH. DeBERTa and the batch-10 LLM score the same source rows for their respective matched policies and all-nine probability ensembles.

This supplements the completed evaluation in Notebook 22. It tests feed availability, not sentiment against a price-only control: no H1-eligible frozen price-only index comparator exists.


In [2]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

CODE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
RESULT_ROOT = (
    CODE_ROOT / "experiments" / "cache" / "q2_sentiment_sensitivity" / "results"
)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = json.loads((RESULT_ROOT / "manifest.json").read_text(encoding="utf-8"))
artifact_sha256 = manifest["artifact_sha256"]
for name, expected in artifact_sha256.items():
    path = RESULT_ROOT / name
    assert path.is_file() and sha256(path) == expected, name
assert manifest["no_fitting"] is True
assert manifest["no_threshold_selection"] is True
assert manifest["no_candidate_selection"] is True

comparison = pd.read_parquet(RESULT_ROOT / "comparison.parquet")
audit = json.loads((RESULT_ROOT / "audit.json").read_text(encoding="utf-8"))
numeric = comparison.select_dtypes(include="number").to_numpy(float)
assert len(comparison) == 4 and np.isfinite(numeric).all()
baseline_error = max(
    max(details["baseline_max_probability_error"].values())
    for details in audit.values()
)
assert baseline_error < 1e-10
VALIDATED = True

The table compares identical policies under the original zero-fresh-feed replay and the fresh-feed replay, sorted by fresh Net. Delta Net is measured in percentage points; trade counts show whether feed availability changes execution.


In [3]:
assert VALIDATED
labels = {
    "usa500_deberta_soft_vote": "DeBERTa all-nine ensemble",
    "usa500_best_single_deberta_svm": "DeBERTa SVM",
    "usatech_deepseek_soft_vote": "LLM all-nine ensemble",
    "usatech_best_single_deepseek_lstm": "LLM LSTM",
}
scorers = {"usa500": "DeBERTa", "usatech": "LLM"}
table = pd.DataFrame({
    "Market": comparison["stream"].map({"usa500": "USA500", "usatech": "USATECH"}),
    "Frozen policy": comparison["candidate_id"].map(labels),
    "Fresh scorer": comparison["stream"].map(scorers),
    "Trades original": comparison["original_trades"].astype(int),
    "Trades fresh": comparison["fresh_trades"].astype(int),
    "L/S fresh": (
        comparison["fresh_long_trades"].astype(int).astype(str)
        + "/"
        + comparison["fresh_short_trades"].astype(int).astype(str)
    ),
    "Net original %": 100 * comparison["original_net_return"],
    "Net fresh %": 100 * comparison["fresh_net_return"],
    "Delta Net pp": 100 * comparison["delta_net_return"],
    "Sharpe": comparison["fresh_daily_sharpe"],
    "Sortino": comparison["fresh_daily_sortino"],
})
table = table.sort_values("Net fresh %", ascending=False).reset_index(drop=True)
required_numeric = [
    "Trades original", "Trades fresh", "Net original %", "Net fresh %",
    "Delta Net pp", "Sharpe", "Sortino",
]
assert np.isfinite(table[required_numeric].to_numpy(float)).all()
display(table.style.format({
    "Net original %": "{:.3f}", "Net fresh %": "{:.3f}",
    "Delta Net pp": "{:+.3f}", "Sharpe": "{:.3f}", "Sortino": "{:.3f}",
}).hide(axis="index"))


Market,Frozen policy,Fresh scorer,Trades original,Trades fresh,L/S fresh,Net original %,Net fresh %,Delta Net pp,Sharpe,Sortino
USA500,DeBERTa all-nine ensemble,DeBERTa,3,4,3/1,0.910,0.937,+0.028,3.064,0.000
USATECH,LLM all-nine ensemble,LLM,2,2,1/1,0.859,0.859,+0.000,2.193,0.000
USATECH,LLM LSTM,LLM,2,2,2/0,1.062,0.334,-0.728,0.739,1.494
USA500,DeBERTa SVM,DeBERTa,1,1,1/0,-0.125,-0.125,+0.000,-2.003,-2.003


## Results

Fresh sentiment adds one SHORT trade to the USA500 ensemble and increases Net by 0.028 percentage points to 0.937%. USATECH LLM LSTM falls by 0.728 percentage points to 0.334%; USA500 DeBERTa SVM and the USATECH ensemble are unchanged. The sample contains only one to four trades per policy.

Takeaway: Fresh Q2 sentiment has no consistent net-of-cost effect across these four frozen policies.
